<font size="10">Purpose</font>

 screens PDF resumes using a trained ML model and exports a ranked Excel report

<font size="10">Import Libraries</font>

In [169]:
import os
import re
import warnings
import argparse
import glob
from pathlib import Path
from datetime import datetime
import pandas as pd
import pandas as pd
import numpy as np
 
# PDF extraction
try:
    import pdfplumber
except ImportError:
    pdfplumber = None
 
# ML
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.metrics import classification_report, accuracy_score, precision_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import recall_score, f1_score 
# Excel output
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
 
warnings.filterwarnings("ignore")

<font size="8">Load training data</font>

 reads training data from CSV or Excel; normalises recruiter decisions to binary (Hire = 1 / Not Hire = 0)

In [170]:
# ─────────────────────────────────────────────
# 1. LOAD & CLEAN TRAINING DATA
# ─────────────────────────────────────────────
 
def load_training_data(path: str) -> pd.DataFrame:
    """Load CSV or Excel training file."""
    ext = Path(path).suffix.lower()
    df = pd.read_excel(path) if ext in (".xlsx", ".xls") else pd.read_csv(path)
    return df

 
def clean_training_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and normalise the training dataframe."""
    df = df.copy()
 
    # Strip whitespace from column names
    df.columns = df.columns.str.strip()
 
    # Normalise recruiter decision → binary
    decision_col = "Recruiter Decision"
    df[decision_col] = (
        df[decision_col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(lambda x: 1 if x in ("hire", "hired", "yes", "1") else 0)
    )
 
    # Fill nulls
    df["Certifications"] = df["Certifications"].fillna("None")
    df["Skills"] = df["Skills"].fillna("")
    df["Education"] = df["Education"].fillna("Unknown")
    df["Job Role"] = df["Job Role"].fillna("Unknown")
    df["Experience (Years)"] = pd.to_numeric(df["Experience (Years)"], errors="coerce").fillna(0)
    df["Projects Count"] = pd.to_numeric(df["Projects Count"], errors="coerce").fillna(0)
    df["AI Score (0-100)"] = pd.to_numeric(df["AI Score (0-100)"], errors="coerce").fillna(50)
 
    return df

<font size="8">FEATURE ENGINEERING</font>

encodes education level (0–5 scale), counts comma-separated skills, flags presence of certifications, and label-encodes job roles

In [171]:
# ─────────────────────────────────────────────
# 2. FEATURE ENGINEERING
# ─────────────────────────────────────────────
 
EDUCATION_RANK = {
    "phd": 5, "ph.d": 5, "doctorate": 5,
    "m.tech": 4, "m.sc": 4, "msc": 4, "mba": 4, "master": 4, "m.e": 4,
    "b.tech": 3, "b.sc": 3, "bsc": 3, "be": 3, "b.e": 3, "bachelor": 3,
    "diploma": 2, "associate": 2,
    "12th": 1, "high school": 1,
    "unknown": 0,
}
 
 
def education_level(edu: str) -> int:
    edu_l = str(edu).lower().strip()
    for key, val in EDUCATION_RANK.items():
        if key in edu_l:
            return val
    return 0
 
 
def skills_count(skills: str) -> int:
    return len([s for s in str(skills).split(",") if s.strip()])
 
 
def has_cert(cert: str) -> int:
    return 0 if str(cert).strip().lower() in ("none", "nan", "", "no") else 1
 
 
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["edu_level"] = df["Education"].apply(education_level)
    df["skills_count"] = df["Skills"].apply(skills_count)
    df["has_cert"] = df["Certifications"].apply(has_cert)
    return df
 

<font size="8">TRAIN MODEL</font>

 trains a RandomForestClassifier (200 estimators, balanced class weights) with an 80/20 train-test split; reports accuracy, precision, and full classification report

In [172]:
# ─────────────────────────────────────────────
# 3. TRAIN MODEL
# ─────────────────────────────────────────────

def train_model(df: pd.DataFrame):
    df = build_features(df)

    le_role = LabelEncoder()
    df["job_role_enc"] = le_role.fit_transform(df["Job Role"].astype(str))

    feature_cols = [
        "Experience (Years)", "Projects Count", #"AI Score (0-100)"-data leakage,  
        "edu_level", "skills_count", "has_cert", "job_role_enc",
    ]
    X = df[feature_cols]
    y = df["Recruiter Decision"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    
    # metrics = {
    #     "accuracy": round(accuracy_score(y_test, y_pred), 4),
    #     "precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
    #     "report": classification_report(y_test, y_pred, target_names=["Not Hired", "Hired"]),
    # }
    
    metrics = {
    "accuracy": round(accuracy_score(y_test, y_pred), 4),
    "precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
    "recall": round(recall_score(y_test, y_pred), 4),
    "f1": round(f1_score(y_test, y_pred), 4),
    "report": classification_report(
        y_test,
        y_pred,
        target_names=["Not Hired", "Hired"]
    ),
    }
    print(f"Recall   : {metrics['recall']}")
    print(f"F1 Score : {metrics['f1']}")
    print("\n===== Model Evaluation =====")
    print(f"Accuracy : {metrics['accuracy']}")
    print(f"Precision: {metrics['precision']}")
    print(metrics["report"])

    clf.fit(X[feature_cols], y)
    return clf, le_role, feature_cols, metrics

 


<font size="8">PDF Parsing</font>

 extracts raw text with pdfplumber; splits into sections (Education, Experience, Skills, Projects, Certifications) using regex-based header detection

In [173]:
KNOWN_SKILLS = [
    "python", "java", "sql", "r", "tensorflow", "pytorch", "keras", "scikit-learn",
    "machine learning", "deep learning", "nlp", "natural language processing",
    "computer vision", "data analysis", "pandas", "numpy", "matplotlib",
    "react", "angular", "vue", "javascript", "typescript", "html", "css",
    "aws", "azure", "gcp", "docker", "kubernetes", "linux", "git",
    "tableau", "power bi", "excel", "spark", "hadoop"
]

def normalize_text(text):
    text = text.lower()

    # remove dots in degrees like m.sc. → msc
    text = text.replace("m.sc.", "msc")
    text = text.replace("b.sc.", "bsc")
    text = text.replace("m.tech", "mtech")
    text = text.replace("b.tech", "btech")

    return text
EDUCATION_KEYWORDS = {
    "PHD": ["phd", "doctorate"],
    "MASTER": ["master", "mba", "m.tech", "msc", "m.sc"],
    "BACHELOR": ["bachelor", "b.tech", "btech", "bsc", "b.sc", "be"],
    "DIPLOMA": ["diploma"]
}

CERT_KEYWORDS = [
    "certified", "certificate", "aws", "google", "microsoft",
    "coursera", "udemy", "pmp", "cissp", "azure"
]

JOB_ROLE_KEYWORDS = {
    "data analyst": [
        "data analyst", "business analyst", "bi analyst",
        "data analysis", "reporting", "dashboard",
        "sql", "excel", "power bi", "tableau",
        "data visualization", "etl", "data cleaning",
        "kpi", "metrics"
    ],

    "data scientist": [
        "data scientist",
        "machine learning", "ml model", "deep learning",
        "nlp", "natural language processing",
        "computer vision",
        "model training", "model evaluation",
        "feature engineering",
        "predictive modeling",
        "classification", "regression", "clustering",
        "neural networks"
    ],

    "software engineer": [
        "software engineer", "software developer", "developer", "programmer",
        "backend", "frontend", "full stack", "fullstack",
        "web developer", "api", "microservices",
        "system design", "distributed systems",
        "java", "c++", "javascript", "react", "node", "spring", "django"
    ],

    "devops engineer": [
        "devops", "ci/cd", "jenkins", "docker", "kubernetes",
        "terraform", "ansible",
        "infrastructure as code", "deployment pipeline",
        "monitoring", "logging"
    ],

    "cloud engineer": [
        "cloud engineer", "aws", "azure", "gcp",
        "cloud architecture", "serverless", "lambda",
        "cloud infrastructure"
    ],

    "ai engineer": [
        "ai engineer", "machine learning engineer", "ml engineer",
        "mlops", "model deployment", "ml pipeline",
        "tensorflow", "pytorch"
    ],

    "product manager": [
        "product manager", "product owner",
        "roadmap", "product strategy",
        "user stories", "stakeholder management"
    ],

    "project manager": [
        "project manager", "project management",
        "pmp", "delivery manager",
        "timeline", "budget", "risk management"
    ],

    "ui ux designer": [
        "ui designer", "ux designer", "ui/ux",
        "figma", "adobe xd",
        "wireframe", "prototyping", "design system"
    ],

    "qa engineer": [
        "qa engineer", "test engineer", "software tester",
        "automation testing", "manual testing",
        "selenium", "test cases"
    ],

    "business analyst": [
        "business analyst",
        "requirement gathering",
        "brd", "frd",
        "gap analysis"
    ],

    "database administrator": [
        "database administrator", "dba",
        "sql server", "oracle", "mysql", "postgresql",
        "query optimization"
    ],

    "cybersecurity analyst": [
        "cybersecurity", "security analyst",
        "penetration testing", "vulnerability assessment",
        "soc", "ethical hacking"
    ],

    "mobile developer": [
        "android developer", "ios developer",
        "kotlin", "swift", "flutter", "react native"
    ],

    "technical support engineer": [
        "technical support", "support engineer",
        "help desk", "troubleshooting", "ticketing"
    ]
}


In [174]:
def extract_pdf_text(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

In [175]:
import re

SECTION_MAP = {
    "education": ["education", "academic", "qualification"],
    "projects": ["projects", "project"],
    "experience": ["experience", "work experience", "employment"],
    "skills": ["skills", "technical skills"],
    "certifications": ["certification", "certifications", "licenses"]
}

import re

def split_sections(text):
    sections = {}
    current_section = None

    for line in text.splitlines():
        clean = line.strip()
        if not clean:
            continue

        lower = clean.lower()

        matched_section = None

        # 🔥 strict header detection
        for section, keywords in SECTION_MAP.items():
            for kw in keywords:
                if re.fullmatch(rf"{kw}", lower):
                    matched_section = section
                    break
            if matched_section:
                break

        if matched_section:
            current_section = matched_section
            sections[current_section] = ""
            continue

        # ✅ accumulate content
        if current_section:
            sections[current_section] += clean + "\n"

    return sections

<font size="8">Parsed Fields</font>

Name, Phone, Email, LinkedIn (from both text and embedded PDF links via PyMuPDF), Skills, Experience (years), Education, Certifications, Projects Count + Names, and inferred Job Role

In [176]:
def parse_name(text):
    lines = [l.strip() for l in text.splitlines() if l.strip()]

    for line in lines[:5]:
        # Remove common prefixes
        line = re.sub(r'^(contact|profile|email|phone|linkedin)\s*[:\-]?\s*', '', line, flags=re.I)

        # Reject if line contains digits (likely phone numbers)
        if re.search(r'\d', line):
            continue

        words = line.split()

        # Check word count and capitalization
        if (2 <= len(words) <= 4 and
            all(w.isalpha() and w[0].isupper() for w in words)):
            return line

    return "Unknown"

In [177]:
def parse_experience(text: str) -> float:
    """Calculate total experience in years (month-level accuracy)."""

    text_l = text.lower()

    total_months = 0

    # 1️⃣ Pattern: "2 years 7 months"
    matches = re.findall(r'(\d+)\s+years?\s*(\d+)?\s*months?', text_l)
    for years, months in matches:
        y = int(years)
        m = int(months) if months else 0
        total_months += y * 12 + m

    # 2️⃣ Pattern: "(8 months)"
    matches = re.findall(r'\((\d+)\s+months?\)', text_l)
    for m in matches:
        total_months += int(m)

    # 3️⃣ Pattern: "2023 - 2025"
    matches = re.findall(r'(20\d{2})\s*[-–]\s*(20\d{2}|present|current)', text_l)
    for start, end in matches:
        start = int(start)
        end = datetime.now().year if end in ["present", "current"] else int(end)
        total_months += (end - start) * 12

    # Avoid double counting → normalize
    if total_months == 0:
        return 0

    return round(total_months / 12, 1)

In [178]:
def parse_skills(text):
    text_l = text.lower()
    skills = [s.title() for s in KNOWN_SKILLS if s in text_l]
    return ", ".join(sorted(set(skills))) if skills else "N/A"

In [179]:
def parse_education(text):
    sections = split_sections(text)
    edu_text = sections.get("education", "")

    print("EDU TEXT RAW:\n", edu_text)  # ✅ debug here

    if not edu_text:
        return "Unknown"

    edu_text = normalize_text(edu_text)

    print("EDU TEXT NORMALIZED:\n", edu_text)  # ✅ debug here

    for degree in ["PHD", "MASTER", "BACHELOR", "DIPLOMA"]:
        for kw in EDUCATION_KEYWORDS[degree]:
            if kw in edu_text:
                return degree

    return "Unknown"

In [180]:
def parse_certifications(text):
    sections = split_sections(text)
    cert = sections.get("certifications", "")

    lines = [l.strip() for l in cert.splitlines() if l.strip()]
    return "; ".join(lines[:5]) if lines else "None"

In [181]:
import re

def parse_projects(text: str) -> tuple:
    sections = split_sections(text)
    proj_text = sections.get("projects", "")

    if not proj_text:
        return 0, "N/A"

    lines = [l.strip() for l in proj_text.splitlines() if l.strip()]

    project_names = []

    for line in lines:
        line_clean = re.sub(r"^[•\-–]\s*", "", line).strip()
        line_lower = line_clean.lower()

        # ❌ skip junk lines
        if any(skip in line_lower for skip in [
            "project activities", "technologies", "tools",
            "skills", "responsibilities"
        ]):
            continue

        # ❌ skip sentences (descriptions)
        if line_clean.endswith("."):
            continue

        if len(line_clean.split()) > 12:
            continue

        if re.search(r"\b(analyze|visualize|utilize|develop|employ)\b", line_lower):
            continue

        # ✅ MUST contain "-" (strong signal of title line)
        if "-" not in line_clean:
            continue

        # ✅ extract title before "-"
        name = line_clean.split("-")[0].strip()

        # ❌ remove weak/noisy lines
        if len(re.findall(r"[A-Z][a-z]+", name)) < 2:
            continue

        if 5 < len(name) < 100:
            project_names.append(name)

    # ✅ remove duplicates
    project_names = list(dict.fromkeys(project_names))

    return len(project_names), ", ".join(project_names[:5]) if project_names else "N/A"

In [182]:
import re
import fitz
#import PyMuPDF
print(fitz.__doc__)

def extract_links_from_pdf(pdf_path):
    links = []

    try:
        doc = fitz.open(pdf_path)

        for page in doc:
            for link in page.get_links():
                uri = link.get("uri", "")
                if uri:
                    links.append(uri)

    except Exception as e:
        print(f"Error reading PDF links: {e}")

    return links

def get_linkedin_from_links(links):
    for link in links:
        if "linkedin.com" in link.lower():
            return link
    return "Not Found"

import re

def parse_contact_info_from_text(text):
    # ---------- EMAIL ----------
    emails = re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", text)
    email = list(dict.fromkeys(emails))[0] if emails else "Not Found"

    # ---------- PHONE ----------
    phones = re.findall(r"(?:\+91[\-\s]?|0)?[6-9]\d{9}", text)
    phone = list(dict.fromkeys(phones))[0] if phones else "Not Found"

    # ---------- LINKEDIN (text) ----------
    linkedin_urls = re.findall(
        r"(https?://(?:www\.)?linkedin\.com/[^\s|]+)",
        text.lower()
    )

    if linkedin_urls:
        linkedin_text = linkedin_urls[0]
    else:
        if "linkedin" in text.lower():
            linkedin_text = "Present (URL not extracted)"
        else:
            linkedin_text = "Not Found"

    return phone, email, linkedin_text
    
def parse_contact_info(text, pdf_path):
    # Step 1: extract from text
    phone, email, linkedin_text = parse_contact_info_from_text(text)

    # Step 2: extract links from PDF
    links = extract_links_from_pdf(pdf_path)
    linkedin_pdf = get_linkedin_from_links(links)

    # Step 3: combine (priority to real link)
    if linkedin_pdf != "Not Found":
        linkedin = linkedin_pdf
    else:
        linkedin = linkedin_text

    return phone, email, linkedin

PyMuPDF 1.27.2.2: Python bindings for the MuPDF 1.27.2 library.
Python 3.13 running on win32 (64-bit).



In [183]:
def infer_role(text):
    text_l = text.lower()
    lines = [l.strip() for l in text_l.splitlines() if l.strip()]

    # --- Step 1: Group into experience blocks using dates ---
    date_pattern = re.compile(r'(20\d{2})|present')

    blocks = []
    current_block = []

    for line in lines:
        if date_pattern.search(line):
            if current_block:
                blocks.append(" ".join(current_block))
                current_block = []
        current_block.append(line)

    if current_block:
        blocks.append(" ".join(current_block))

    # --- Step 2: Sort blocks by recency ---
    def get_latest_year(block):
        years = re.findall(r'20\d{2}', block)
        if "present" in block:
            return 9999
        return max(map(int, years)) if years else 0

    blocks = sorted(blocks, key=get_latest_year, reverse=True)

    # --- Step 3: Weighted scoring ---
    for block in blocks[:3]:  # check top 3 recent roles
        role_scores = {}

        for role, kws in JOB_ROLE_KEYWORDS.items():
            score = 0
            for kw in kws:
                if kw in block:
                    # Weighting:
                    # phrases (>=2 words) → stronger signal
                    if " " in kw:
                        score += 3
                    else:
                        score += 1

            if score > 0:
                role_scores[role] = score

        if role_scores:
            best_role, best_score = max(role_scores.items(), key=lambda x: x[1])

            # --- Step 4: Minimum confidence threshold ---
            if best_score >= 3:
                return best_role.title()

    return "General"

<font size="8">ATS Scoring</font>

computes a 0–100 score from skills count, experience tier, project count, education level, certifications, and contact completeness

In [184]:
import re

def compute_ats_score(data):
    score = 0

    # ---------- 1. SKILLS (0–30) ----------
    skills = data.get("Skills", "")
    skill_count = len(skills.split(",")) if skills else 0
    score += min(skill_count * 3, 30)

    # ---------- 2. EXPERIENCE (0–20) ----------
    exp = data.get("Experience (Years)", 0)
    if exp:
        if exp >= 5:
            score += 20
        elif exp >= 3:
            score += 15
        elif exp >= 1:
            score += 5
        else:
            score += 0

    # ---------- 3. PROJECTS (0–15) ----------
    proj_count = data.get("Projects Count", 0)
    score += min(proj_count * 5, 5)

    # ---------- 4. EDUCATION (0–15) ----------
    edu = data.get("Education", "")
    if edu == "PHD":
        score += 15
    elif edu == "MASTER":
        score += 12
    elif edu == "BACHELOR":
        score += 10
    elif edu == "DIPLOMA":
        score += 10

    # ---------- 5. CERTIFICATIONS (0–10) ----------
    certs = data.get("Certifications", "")
    if certs and certs != "None":
        cert_count = len(certs.split(","))
        score += min(cert_count * 2, 10)

    # ---------- 6. CONTACT COMPLETENESS (0–10) ----------
    if data.get("Email") != "Not Found":
        score += 4
    if data.get("Phone") != "Not Found":
        score += 3
    if data.get("LinkedIn") not in ["Not Found", "Present (URL not extractable)"]:
        score += 3

    return min(score, 100)

In [185]:
def parse_resume(pdf_path):
    text = extract_pdf_text(pdf_path)

    proj_count, proj_names = parse_projects(text)
    phone, email, linkedin = parse_contact_info(text, pdf_path)

    data = {
        "File": Path(pdf_path).name,
        "Name": parse_name(text),
        "Skills": parse_skills(text),
        "Experience (Years)": parse_experience(text),
        "Education": parse_education(text),
        "Certifications": parse_certifications(text),
        "Job Role": infer_role(text),
        "Projects Count": proj_count,
        "Project Names": proj_names,
        "Phone": phone,
        "Email": email,
        "LinkedIn": linkedin,
        "_raw_text": text
    }

    # 🔥 compute ATS score
    data["AI Score (0-100)"] = compute_ats_score(data)

    return data

In [186]:
import os
import pandas as pd

def process_folder(folder="resumes"):
    results = []
    failed_files = []

    for root, _, files in os.walk(folder):  # ✅ supports subfolders
        for file in files:
            if not file.lower().endswith(".pdf"):
                continue

            path = os.path.join(root, file)

            try:
                # 🔥 pass correct variable
                data = parse_resume(path)

                # ✅ attach filename
                data["File"] = file

                print(f"✔ {file} → {data.get('Name', 'Unknown')}")

                results.append(data)

            except Exception as e:
                print(f"✘ {file}: {e}")
                failed_files.append({
                    "File": file,
                    "Error": str(e)
                })

    df = pd.DataFrame(results)

    # ✅ ensure consistent column order (optional but clean)
    columns_order = [
        "File", "Name", "Skills", "Experience", "Education",
        "Certifications", "Role", "Projects Count", "Project Names",
        "Phone", "Email", "LinkedIn"
    ]

    df = df.reindex(columns=columns_order)

    # 🔍 optional: show failed files
    if failed_files:
        print("\n⚠ Failed Files:")
        for f in failed_files:
            print(f)

    return df

<font size="8">Prediction</font>

runs trained classifier on parsed candidates; outputs Hire / Not Hire label and hire probability

In [187]:
# ─────────────────────────────────────────────
# 6. PREDICT & SCORE
# ─────────────────────────────────────────────
 
def predict_candidates(clf, le_role, feature_cols, candidates: pd.DataFrame) -> pd.DataFrame:
    df = candidates.copy()
    df = build_features(df)
 
    # Encode job role (handle unseen labels)
    known_roles = set(le_role.classes_)
    df["job_role_enc"] = df["Job Role"].apply(
        lambda r: le_role.transform([r])[0] if r in known_roles else 0
    )
 
    X = df[feature_cols]
    df["Predicted"] = clf.predict(X)
    df["Hire Probability"] = clf.predict_proba(X)[:, 1].round(3)
    df["Predicted Label"] = df["Predicted"].map({1: "Hire", 0: "Not Hire"})
 
    # Composite rank score
    df["Rank Score"] = (
        df["Hire Probability"] * 40 +
        df["Experience (Years)"].clip(0, 15) / 15 * 20 +
        df["edu_level"] / 5 * 15 +
        df["skills_count"].clip(0, 20) / 20 * 15 +
        df["has_cert"] * 5 +
        df["Projects Count"].clip(0, 10) / 10 * 5
    ).round(2)
 
    return df

<font size="8">Auto Comments</font>

generates plain-English profile summaries (e.g. "Highly experienced; Broad skill set; No certifications")

In [188]:
# ─────────────────────────────────────────────
# 7. GENERATE COMMENTS
# ─────────────────────────────────────────────
 
def generate_comment(row: pd.Series) -> str:
    comments = []
    if row["Experience (Years)"] < 2:
        comments.append("Less experience")
    elif row["Experience (Years)"] >= 8:
        comments.append("Highly experienced")
 
    if row["edu_level"] <= 2:
        comments.append("Lower educational qualification")
    elif row["edu_level"] >= 4:
        comments.append("Strong educational background")
 
    if row["skills_count"] < 3:
        comments.append("Limited skill set")
    elif row["skills_count"] >= 8:
        comments.append("Broad skill set")
 
    if row["has_cert"] == 0:
        comments.append("No certifications")
    else:
        comments.append("Has relevant certifications")
 
    if row["Projects Count"] == 0:
        comments.append("No projects listed")
    elif row["Projects Count"] >= 5:
        comments.append("Strong project portfolio")
 
    return "; ".join(comments) if comments else "Profile meets basic criteria"

<font size="8">Ranking</font>

 calculates a composite Rank Score (hire probability + experience + education + skills + certs + projects) and sorts candidates descending

In [189]:
# ─────────────────────────────────────────────
# 8. RANK & SELECT TOP-N
# ─────────────────────────────────────────────
 
def rank_candidates(df: pd.DataFrame, top_n: int = None) -> pd.DataFrame:
    df = df.copy()
    df["Comment"] = df.apply(generate_comment, axis=1)
    df = df.sort_values("Rank Score", ascending=False)
    df["Rank"] = range(1, len(df) + 1)
    if top_n:
        df = df.head(top_n)
    return df

<font size="8">Excel Export</font>

The generated Excel file contains:


Summary — candidate counts and predicted hires per job role

Per-Role Sheets — top candidates filtered by role, colour-coded green (Hire) / red (Not Hire)

All Candidates — complete ranked list across all roles

In [190]:
# ─────────────────────────────────────────────
# 9. EXPORT TO EXCEL (multi-tab)
# ─────────────────────────────────────────────
 
OUTPUT_COLS = [
    "Rank", "AI Score (0-100)", "Name", "Job Role", "Skills", "Experience (Years)",
    "Education", "Certifications", "Projects Count", "Project Names",
    "Predicted Label", "Hire Probability", "Comment",  "Phone", "Email","LinkedIn"
]
 
HEADER_FILL = PatternFill("solid", start_color="1F4E79")
HIRE_FILL = PatternFill("solid", start_color="C6EFCE")
NOTHIRE_FILL = PatternFill("solid", start_color="FFCCCC")
ALT_FILL = PatternFill("solid", start_color="EBF3FB")
HEADER_FONT = Font(bold=True, color="FFFFFF", name="Arial", size=10)
CELL_FONT = Font(name="Arial", size=10)
THIN = Side(style="thin", color="B0B0B0")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
 
 
def _write_sheet(ws, df: pd.DataFrame, cols: list):
    ws.append(cols)
    for cell in ws[1]:
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = BORDER
    ws.row_dimensions[1].height = 30
 
    for i, (_, row) in enumerate(df[cols].iterrows(), start=2):
        ws.append([row[c] if c in row.index else "" for c in cols])
        is_hire = str(row.get("Predicted Label", "")).lower() == "hire"
        row_fill = HIRE_FILL if is_hire else (NOTHIRE_FILL if not is_hire else ALT_FILL)
        if i % 2 == 0 and not is_hire:
            row_fill = ALT_FILL
        for cell in ws[i]:
            cell.font = CELL_FONT
            cell.alignment = Alignment(vertical="center", wrap_text=True)
            cell.border = BORDER
            if is_hire:
                cell.fill = HIRE_FILL
 
    # Auto-width
    for col_idx, col_name in enumerate(cols, start=1):
        max_len = max(
            len(str(col_name)),
            *[len(str(ws.cell(row=r, column=col_idx).value or "")) for r in range(2, ws.max_row + 1)],
        )
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 4, 50)
 
 
def export_excel(df: pd.DataFrame, output_path: str, top_n: int = None):
    """Export candidates to multi-tab Excel grouped by Job Role."""
    wb = Workbook()
    wb.remove(wb.active)  # Remove default sheet
 
    # ── Summary tab ──────────────────────────────────────────────────────
    ws_summary = wb.create_sheet("Summary")
    summary_roles = df["Job Role"].value_counts().reset_index()
    summary_roles.columns = ["Job Role", "Total Candidates"]
    hire_counts = df[df["Predicted Label"] == "Hire"].groupby("Job Role").size().reset_index()
    hire_counts.columns = ["Job Role", "Predicted Hires"]
    summary = summary_roles.merge(hire_counts, on="Job Role", how="left").fillna(0)
    summary["Predicted Hires"] = summary["Predicted Hires"].astype(int)
    _write_sheet(ws_summary, summary, ["Job Role", "Total Candidates", "Predicted Hires"])
    ws_summary.title = "Summary"
 
    # ── Per Job Role tabs ─────────────────────────────────────────────────
    for role in sorted(df["Job Role"].unique()):
        role_df = df[df["Job Role"] == role].copy()
        if top_n:
            role_df = role_df.head(top_n)
        safe_name = re.sub(r"[\\/*?:\[\]]", "_", role)[:31]
        ws = wb.create_sheet(safe_name)
        cols_to_write = [c for c in OUTPUT_COLS if c in role_df.columns]
        _write_sheet(ws, role_df, cols_to_write)
 
    # ── All Candidates tab ────────────────────────────────────────────────
    ws_all = wb.create_sheet("All Candidates")
    cols_to_write = [c for c in OUTPUT_COLS if c in df.columns]
    _write_sheet(ws_all, df, cols_to_write)
 
    wb.save(output_path)
    print(f"\n✅ Excel report saved → {output_path}")

In [191]:
# ─────────────────────────────────────────────
# 10. MAIN PIPELINE
# ─────────────────────────────────────────────
 
def run_pipeline(
    training_file: str,
    resume_folder: str,
    output_excel: str = "screening_report.xlsx",    #change #output file name
    top_n: int = None,
):
    print("=" * 55)
    print("  SMART RESUME SCREENING PIPELINE")
    print("=" * 55)
 
    # ── Step 1-2: Load & clean training data ──────────────────────────
    print("\n[1/5] Loading training data…")
    train_df = load_training_data(training_file)
    train_df = clean_training_data(train_df)
    print(f"      {len(train_df)} training records loaded.")
 
    # ── Step 3-5: Train model ─────────────────────────────────────────
    print("[2/5] Training classifier…")
    clf, le_role, feature_cols, metrics = train_model(train_df)
 
    # ── Step 6-7: Extract resumes ─────────────────────────────────────
    pdf_files = glob.glob(os.path.join(resume_folder, "**/*.pdf"), recursive=True) + \
                glob.glob(os.path.join(resume_folder, "*.pdf"))
    pdf_files = list(set(pdf_files))
 
    if not pdf_files:
        print(f"\n⚠️  No PDF files found in: {resume_folder}")
        return
 
    print(f"[3/5] Extracting data from {len(pdf_files)} PDF resume(s)…")
    parsed = []
    for path in pdf_files:
        try:
            data = parse_resume(path)
            parsed.append(data)
            print(f"      ✔ {Path(path).name}  →  {data['Name']}")
        except Exception as e:
            print(f"      ✘ {Path(path).name}: {e}")
 
    if not parsed:
        print("No resumes successfully parsed. Exiting.")
        return
 
    candidates_df = pd.DataFrame(parsed)
 
    # ── Step 8-10: Predict ────────────────────────────────────────────
    print("[4/5] Predicting hire decisions…")
    results_df = predict_candidates(clf, le_role, feature_cols, candidates_df)
 
    # ── Step 11-12: Rank & comment ────────────────────────────────────
    results_df = rank_candidates(results_df, top_n=top_n)
    print(f"      Ranked {len(results_df)} candidate(s).")

    # Show preview
    preview_cols = ["Rank", "Name", "Job Role", "Predicted Label", "Hire Probability", "Comment" ]
    print("\n── Candidate Preview ──────────────────────────────────")
    print(results_df[preview_cols].to_string(index=False))
 
    # ── Step 13: Export Excel ─────────────────────────────────────────
    print("\n[5/5] Exporting Excel report…")
    export_excel(results_df, output_excel, top_n=top_n)
 
    return results_df
 
 
# ─────────────────────────────────────────────
# CLI ENTRY POINT
# ─────────────────────────────────────────────
# CLI / JUPYTER ENTRY POINT
# ─────────────────────────────────────────────

import sys

if 'ipykernel' in sys.modules:
    # ── Running in Jupyter — set your paths directly here ──
    training_file  = r"D:\Project\Resume screening\AI_Resume_Screening.csv"   #change #training file name
    resume_folder  = r"D:\Project\Resume screening\resumes"                   #change #put the resumes in resumes folder
    output_excel   = r"D:\Project\Resume screening\screening_report.xlsx"     #change #output file
    top_n          = 10  # or None for all candidates                         #change #TOP N 

    run_pipeline(
        training_file=training_file,
        resume_folder=resume_folder,
        output_excel=output_excel,
        top_n=top_n,
    )

else:
    # ── Running from terminal / command line ──
    parser = argparse.ArgumentParser(description="Smart Resume Screening Tool")
    parser.add_argument("--training", required=True, help="Path to training CSV/Excel file")
    parser.add_argument("--resumes",  required=True, help="Folder containing PDF resumes")
    parser.add_argument("--output",   default="screening_report.xlsx", help="Output Excel path")
    parser.add_argument("--top_n",    type=int, default=None, help="Top N resumes per role")
    args = parser.parse_args()

    run_pipeline(
        training_file=args.training,
        resume_folder=args.resumes,
        output_excel=args.output,
        top_n=args.top_n,
    )
 


  SMART RESUME SCREENING PIPELINE

[1/5] Loading training data…
      1000 training records loaded.
[2/5] Training classifier…
Recall   : 0.9753
F1 Score : 0.9814

===== Model Evaluation =====
Accuracy : 0.97
Precision: 0.9875
              precision    recall  f1-score   support

   Not Hired       0.90      0.95      0.92        38
       Hired       0.99      0.98      0.98       162

    accuracy                           0.97       200
   macro avg       0.94      0.96      0.95       200
weighted avg       0.97      0.97      0.97       200

[3/5] Extracting data from 9 PDF resume(s)…
EDU TEXT RAW:
 2024-2026 Bachelor’s of Computer Applications Guru Gobind Singh Indraprastha university
(GPA: 8.0)

EDU TEXT NORMALIZED:
 2024-2026 bachelor’s of computer applications guru gobind singh indraprastha university
(gpa: 8.0)

      ✔ DA Resume.pdf  →  Nishant Tyagi
EDU TEXT RAW:
 National Institute of Technology Puducherry
Bachelor of Technology - BTech, Mechanical Engineering · (2017 - 2